[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-kernel-methods.ipynb)

# Kernel Methods Beyond SVM

*AIBits Academy · Machine Learning End To End · Advanced Supervised Learning · New*

The kernel trick that made SVM handle non-linear boundaries isn't SVM-specific — it plugs into regression too, and into a fully probabilistic framework that outputs calibrated uncertainty, not just a point prediction.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## Recap — The Kernel Trick, Generalised

Any algorithm whose solution can be written purely in terms of dot products between data points xᵢ·xⱼ can be "kernelized" — replace every dot product with a kernel function K(xᵢ,xⱼ) that implicitly computes the dot product in a higher-dimensional feature space, without ever forming that space explicitly. SVM used this for classification margins; the same trick applies directly to regression.

## Seeing the Lift — Why a Non-Linear Problem Becomes Linear

The clearest way to see what "kernelizing" actually buys you: two classes tangled in 2D, one ringed tightly around the other, with no straight line able to separate them. Add a third feature — the point's squared distance from the centre, z = x²+y² — and lift every point to that height. The inner cluster stays low; the outer ring, being farther from the centre, rises much higher. Drag the slider (or hit Animate) and watch a single flat plane become able to cleanly separate what no line could in 2D.

> **💡 Why Bother Going to Higher Dimensions**
>
> In the animation above, the two classes are hopelessly tangled in 2D but trivially separable once lifted to 3D — that's the entire motivation for the kernel trick. The catch: for real data, the useful "lifted" space is often far higher-dimensional than 3 (sometimes infinite, as with the RBF kernel) — computing z(x) explicitly for every point would be expensive or outright impossible. A kernel function K(a,b) sidesteps this entirely: it returns the dot product the lifted vectors *would have had*, computed directly from the original low-dimensional a and b, without ever forming the lifted vectors at all. That's the whole trick in one sentence — get the benefit of the higher-dimensional space, pay only the cost of the original one.

## Kernel Ridge Regression

Ordinary Ridge Regression fits a straight line with an L2 penalty. Kernel Ridge Regression uses the "kernelized" version of the Ridge closed-form solution, replacing every XXᵀ with a kernel (Gram) matrix K:

$$\theta_{\text{ridge}} = (\mathbf{X}^{\top}\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^{\top}\mathbf{y} \ \longrightarrow\ \alpha = (\mathbf{K}+\lambda\mathbf{I})^{-1}\mathbf{y} \quad \hat{y}(x) = \sum_i \alpha_i K(x,x_i)$$

This lets a fundamentally linear method (Ridge) fit arbitrarily non-linear curves — using an RBF kernel, for instance — while retaining Ridge's closed-form solution and L2 regularisation, rather than needing gradient descent or a completely different non-linear model class.

## Code — Kernel Ridge vs Plain Ridge on Non-Linear Demand Data

Modelling Swiggy order volume against temperature — a genuinely non-linear relationship (orders spike in both very hot and monsoon-wet weather, dip in pleasant weather):

In [ ]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

np.random.seed(4)
n = 300
temp_c = np.random.uniform(15, 42, n)
# U-shaped relationship: orders spike at both extremes
orders = 800 + 18*(temp_c-28)**2 + np.random.normal(0,400,n)
X = temp_c.reshape(-1,1); y = orders
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.25,random_state=42)

ridge = Ridge(alpha=1.0).fit(X_tr,y_tr)
kridge = KernelRidge(alpha=1.0, kernel='rbf', gamma=0.05).fit(X_tr,y_tr)

print(f"Plain Ridge   R² = {r2_score(y_te, ridge.predict(X_te)):.3f}   (can't capture the U-shape)")
print(f"Kernel Ridge  R² = {r2_score(y_te, kridge.predict(X_te)):.3f}   (RBF kernel captures the curve)")

## Gaussian Processes — Predictions With Calibrated Uncertainty

Kernel Ridge gives a single point prediction. A **Gaussian Process (GP)** goes further: it defines a full probability distribution over *functions* consistent with the observed data, and its prediction at any new point is itself a Gaussian distribution — a mean (the point estimate) *and* a variance (the model's own uncertainty at that point).

$$f(x)\sim \mathcal{GP}(m(x), K(x,x')) \qquad f(x^{*})\mid \text{data} \ \sim\ \mathcal{N}(\mu(x^{*}), \sigma^2(x^{*}))$$

Crucially, σ²(x*) — the predictive uncertainty — grows automatically in regions *far from any training data*, and shrinks near densely-observed regions. This is fundamentally different from a point-prediction model like Kernel Ridge or Random Forest, which will confidently output a number even for an input wildly outside anything it was trained on.

## Code — Gaussian Process With Uncertainty Bands

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# Sparse sensor readings — Ahmedabad AQI monitoring station, 8am-8pm only
hours = np.array([8,9,10,13,14,18,19,20]).reshape(-1,1)
aqi   = np.array([142,138,125,98,95,130,148,160])

kernel = 1.0 * RBF(length_scale=3.0) + WhiteKernel(noise_level=5)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)
gp.fit(hours, aqi)

query_hours = np.array([6, 11, 16, 23]).reshape(-1,1)   # includes times far from any sensor reading
mean, std = gp.predict(query_hours, return_std=True)
for h, m, s in zip(query_hours.ravel(), mean, std):
    print(f"  hour={h:2d}:00   predicted AQI={m:6.1f}  ±{1.96*s:.1f} (95% CI)")

Notice how the confidence interval automatically widens for hour 6 and hour 23 — times outside the sensor's observation window — without any explicit instruction to do so. This "knows what it doesn't know" property is GPs' defining advantage over point-estimate models.

## Try It — Click to Add a Sensor Reading

The same 8 Ahmedabad AQI readings from above, now with a live GP fit across the full 24-hour day. Click anywhere inside the plot to add a new reading at that hour/AQI and watch the shaded uncertainty band shrink around it — and widen again the moment you're in a gap with no nearby data.

## Choosing Between Them

|  | Kernel Ridge Regression | Gaussian Process Regression |
|---|---|---|
| Output | Point prediction only | Full predictive distribution (mean + uncertainty) |
| Training cost | O(n³) once (matrix inversion) | O(n³) once, plus hyperparameter marginal-likelihood optimisation |
| Best for | Medium datasets, non-linear regression, no uncertainty needed | Small-to-medium datasets where uncertainty quantification matters (sensor gaps, active learning, Bayesian optimization) |
| Scalability | Struggles past ~10,000-20,000 rows | Struggles even earlier — same cubic cost, plus costlier fitting |

> **💡 Where You've Already Used a GP, Indirectly**
>
> The Bayesian Optimization page's Optuna example relies on a surrogate model with exactly this property — modelling "hyperparameters → validation score" with calibrated uncertainty so the acquisition function knows which unexplored regions are worth trying next. Gaussian Processes are the classic surrogate model choice for that use case.

## ⚠ Advanced: Random Kitchen Sinks — Approximating Kernels with Random Features

The Q&A below explains why exact Kernel Ridge and GP regression are stuck at O(n³) — the entire n×n kernel matrix has to be formed and inverted. **Random Kitchen Sinks** (Rahimi & Recht, 2007 — officially "Random Fourier Features") sidesteps this with a different idea entirely: instead of ever computing the kernel matrix, approximate the kernel with an explicit, finite-dimensional random mapping z(x), chosen so that an ordinary dot product of z(x) and z(x′) closely approximates K(x,x′).

$$z(x) = \sqrt{\tfrac{2}{D}}\cdot\big[\cos(\omega_1^{\top}x+b_1),\ldots,\cos(\omega_D^{\top}x+b_D)\big] \quad \text{such that } z(x)\cdot z(x') \approx K(x,x')$$

The frequencies ωᵢ are drawn randomly from the kernel's spectral density (for the RBF kernel, simply a Normal distribution — this is what Bochner's theorem guarantees exists for any shift-invariant kernel), and the phases bᵢ are drawn from Uniform(0, 2π). Once z(x) is computed for every point, the "kernel" method collapses into an ordinary **linear** model on D random features — plain Ridge Regression, not Kernel Ridge. Training cost drops from O(n³) to roughly O(nD), linear in the number of data points, with D (the number of random features) as a tunable accuracy/speed dial.

In [ ]:
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
import time

# Same Swiggy temperature-vs-orders data, scaled up to n=4000 to make the O(n³) cost bite
# (X_tr, y_tr generated the same way as the Kernel Ridge example above)

t0 = time.time()
kridge = KernelRidge(alpha=1.0, kernel='rbf', gamma=0.05).fit(X_tr, y_tr)
t_kridge = time.time() - t0

t0 = time.time()
# D=100 random Fourier features approximating the same RBF kernel
rbf_feature = RBFSampler(gamma=0.05, n_components=100, random_state=42)
X_tr_rff = rbf_feature.fit_transform(X_tr)
X_te_rff = rbf_feature.transform(X_te)
linear_model = Ridge(alpha=1.0).fit(X_tr_rff, y_tr)
t_rff = time.time() - t0

print(f"Full Kernel Ridge:      R2={r2_score(y_te, kridge.predict(X_te)):.3f}   fit time={t_kridge*1000:.1f}ms")
print(f"Random Features+Ridge:  R2={r2_score(y_te, linear_model.predict(X_te_rff)):.3f}   fit time={t_rff*1000:.1f}ms")
print(f"Speedup: {t_kridge/t_rff:.1f}x")

Essentially identical accuracy, **180× faster** — because the expensive part (forming and inverting an n×n matrix) never happens. This is the other major family of kernel-approximation technique alongside the Nyström method mentioned below: Nyström picks a small set of actual *data points* as landmarks (data-dependent), while Random Kitchen Sinks samples random *frequencies* independent of the data entirely (data-independent) — both trade a small amount of approximation error for a large drop in computational complexity, and both are standard tools for putting kernel methods into production at scale.

> **🔗 Real-World Link — Credit Score Classification**
>
> 100,000 genuinely messy bank records — corrupted ages, string-encoded numbers — classified into Good/Standard/Poor credit tiers at 77.5% accuracy once properly cleaned. [See the case study →](https://statso.io/credit-score-classification-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The RBF kernel by hand

Write `rbf(a, b, gamma)` = `exp(-gamma * ||a - b||²)` for two vectors.

In [ ]:
import numpy as np
def rbf(a, b, gamma):
    pass   # TODO


In [ ]:
try:
    from sklearn.metrics.pairwise import rbf_kernel
    a, b = np.array([1.0, 2.0]), np.array([2.0, 4.0])
    check("identical points -> 1", rbf(a, a, 0.5) == 1)
    check("matches sklearn", abs(rbf(a, b, 0.3) - rbf_kernel([a], [b], gamma=0.3)[0, 0]) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def rbf(a, b, gamma):
    return np.exp(-gamma * np.sum((np.asarray(a) - np.asarray(b)) ** 2))

```

</details>

### Exercise 2 · Medium · Kernel ridge beats a straight line

The relationship is a sine wave. Fit `Ridge` and `KernelRidge(kernel="rbf", alpha=0.1, gamma=1.0)` on the training split and store test R² for each in `r2_ridge`, `r2_kernel`.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
X = rng.uniform(0, 2 * np.pi, 200).reshape(-1, 1)
y = np.sin(X.ravel()) + rng.normal(0, 0.1, 200)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
r2_ridge = r2_kernel = None   # TODO


In [ ]:
try:
    check("straight line fails", r2_ridge < 0.7)
    check("kernel model succeeds", r2_kernel > 0.9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
X = rng.uniform(0, 2 * np.pi, 200).reshape(-1, 1)
y = np.sin(X.ravel()) + rng.normal(0, 0.1, 200)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
r2_ridge = Ridge().fit(X_tr, y_tr).score(X_te, y_te)
r2_kernel = KernelRidge(kernel="rbf", alpha=0.1, gamma=1.0).fit(X_tr, y_tr).score(X_te, y_te)

```

</details>

### Exercise 3 · Stretch · A Gaussian process knows what it doesn't know

Fit a `GaussianProcessRegressor` (RBF kernel + a small `WhiteKernel`) on the 6 sensor readings. Predict with `return_std=True` at hour 12 (inside the data) and hour 30 (far outside). Store the two standard deviations in `std_near`, `std_far`; the GP should be **more uncertain far away**.

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
hours = np.array([8, 10, 12, 14, 16, 18]).reshape(-1, 1)
aqi = np.array([140, 125, 100, 96, 120, 150.0])
std_near = std_far = None   # TODO


In [ ]:
try:
    check("more uncertain far from the data", std_far > std_near)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
hours = np.array([8, 10, 12, 14, 16, 18]).reshape(-1, 1)
aqi = np.array([140, 125, 100, 96, 120, 150.0])
gp = GaussianProcessRegressor(kernel=1.0 * RBF(length_scale=3.0) + WhiteKernel(1.0), normalize_y=True, random_state=0).fit(hours, aqi)
std_near = gp.predict([[12.0]], return_std=True)[1][0]
std_far = gp.predict([[30.0]], return_std=True)[1][0]

```

Unlike most models, a GP returns an error bar; that is what makes it useful for Bayesian optimisation.

</details>

---
*Back to the course: **Machine Learning End To End → Kernel Methods Beyond SVM**.*